# Weights & Biases Integration

This notebook covers **Part 2 (xii)** of the assignment on advanced deep learning customization.

**Topics covered:**
- **W&B Setup**: Project initialization, logging configuration
- **Experiment Tracking**: Metrics, hyperparameters, model artifacts
- **Visualization**: Training curves, confusion matrices, sample predictions
- **Hyperparameter Sweeps**: Automated hyperparameter optimization
- **Model Versioning**: Saving and loading models with W&B Artifacts

**Framework:** TensorFlow/Keras with Weights & Biases

**Dataset:** Fashion MNIST

In [1]:
!pip install -q wandb

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint

print(f"TensorFlow: {tf.__version__}")
print(f"W&B: {wandb.__version__}")

TensorFlow: 2.19.0
W&B: 0.25.1


In [2]:
# Login to W&B (will prompt for API key or use cached credentials)
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shreram25 (shreram25-san-jose-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# Load Fashion MNIST
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
X_train, X_test = X_train / 255.0, X_test / 255.0

X_val, y_val = X_train[:5000], y_train[:5000]
X_train, y_train = X_train[5000:], y_train[5000:]

CLASS_NAMES = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Boot']

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


---
## Part 1: Basic W&B Integration

In [4]:
# Initialize W&B run with configuration
config = {
    'learning_rate': 0.001,
    'epochs': 10,
    'batch_size': 64,
    'optimizer': 'adam',
    'architecture': 'MLP',
    'hidden_units': [128, 64],
    'dropout_rate': 0.3,
    'dataset': 'fashion_mnist'
}

run = wandb.init(
    project='cmpe258-assignment4',
    name='basic-mlp-experiment',
    config=config,
    tags=['baseline', 'mlp']
)

In [5]:
def create_model(config):
    """Creates model based on W&B config."""
    model = keras.Sequential([
        layers.Flatten(input_shape=(28, 28)),
        layers.Dense(config['hidden_units'][0], activation='relu'),
        layers.Dropout(config['dropout_rate']),
        layers.Dense(config['hidden_units'][1], activation='relu'),
        layers.Dropout(config['dropout_rate']),
        layers.Dense(10, activation='softmax')
    ])
    return model

model = create_model(wandb.config)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=wandb.config['learning_rate']),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [7]:
# Train with W&B callbacks
history = model.fit(
    X_train, y_train,
    epochs=wandb.config['epochs'],
    batch_size=wandb.config['batch_size'],
    validation_data=(X_val, y_val),
    callbacks=[
        WandbMetricsLogger(),  # Logs metrics automatically
        WandbModelCheckpoint('models/best_model.keras')  # Saves model checkpoints
    ]
)

Epoch 1/10
860/860 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - accuracy: 0.7548 - loss: 0.6887 - val_accuracy: 0.8516 - val_loss: 0.4055
Epoch 2/10
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8289 - loss: 0.4795 - val_accuracy: 0.8646 - val_loss: 0.3742
Epoch 3/10
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8445 - loss: 0.4330 - val_accuracy: 0.8754 - val_loss: 0.3538
Epoch 4/10
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8537 - loss: 0.4081 - val_accuracy: 0.8752 - val_loss: 0.3453
Epoch 5/10
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8569 - loss: 0.3892 - val_accuracy: 0.8760 - val_loss: 0.3394
Epoch 6/10
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8629 - loss: 0.3765 - val_accuracy: 0.8824 - val_loss: 0.3291
Epoch 7/10
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8674 - loss: 0.3638 - val_accuracy: 0.8792 - val_loss: 0.3331
Epoch 8/10
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8707 - loss: 0.3546 - val_accuracy: 

In [8]:
# Evaluate and log final metrics
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
wandb.log({'test_loss': test_loss, 'test_accuracy': test_acc})
print(f"Test accuracy: {test_acc:.4f}")

Test accuracy: 0.8748


In [9]:
# Finish the run
wandb.finish()

epoch/accuracy,▁▅▆▇▇▇████
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▄▃▂▂▂▁▁▁▁
epoch/val_accuracy,▁▄▆▆▆▇▆▇██
epoch/val_loss,█▅▄▃▃▂▂▃▁▁
test_accuracy,▁
test_loss,▁
epoch/accuracy,0.87551
epoch/epoch,9
epoch/learning_rate,0.001


---
## Part 2: Advanced Logging - Confusion Matrix and Sample Predictions

In [10]:
# Start new run for advanced logging
run = wandb.init(
    project='cmpe258-assignment4',
    name='advanced-logging-experiment',
    config=config
)

In [11]:
# Recreate and train model
model = create_model(wandb.config)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=wandb.config['learning_rate']),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[WandbMetricsLogger()],
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.7643 - loss: 0.6606 - val_accuracy: 0.8486 - val_loss: 0.4208
Epoch 2/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8319 - loss: 0.4691 - val_accuracy: 0.8678 - val_loss: 0.3639
Epoch 3/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8470 - loss: 0.4264 - val_accuracy: 0.8758 - val_loss: 0.3498
Epoch 4/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8539 - loss: 0.4033 - val_accuracy: 0.8760 - val_loss: 0.3373
Epoch 5/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8599 - loss: 0.3886 - val_accuracy: 0.8748 - val_loss: 0.3406


In [12]:
# Log confusion matrix
from sklearn.metrics import confusion_matrix

y_pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)

# Log as W&B plot
wandb.log({
    'confusion_matrix': wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_test,
        preds=y_pred,
        class_names=CLASS_NAMES
    )
})

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [13]:
# Log sample predictions as images
sample_indices = np.random.choice(len(X_test), 16, replace=False)
sample_images = X_test[sample_indices]
sample_labels = y_test[sample_indices]
sample_preds = y_pred[sample_indices]

# Create W&B image table
columns = ['Image', 'True Label', 'Predicted', 'Correct']
table = wandb.Table(columns=columns)

for img, true, pred in zip(sample_images, sample_labels, sample_preds):
    table.add_data(
        wandb.Image(img),
        CLASS_NAMES[true],
        CLASS_NAMES[pred],
        '✓' if true == pred else '✗'
    )

wandb.log({'sample_predictions': table})

In [14]:
# Log precision-recall curve
y_probs = model.predict(X_test)

wandb.log({
    'pr_curve': wandb.plot.pr_curve(
        y_true=y_test,
        y_probas=y_probs,
        labels=CLASS_NAMES
    )
})

313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step


In [15]:
# Log ROC curve
wandb.log({
    'roc_curve': wandb.plot.roc_curve(
        y_true=y_test,
        y_probas=y_probs,
        labels=CLASS_NAMES
    )
})

In [16]:
wandb.finish()

epoch/accuracy,▁▆▇██
epoch/epoch,▁▃▅▆█
epoch/learning_rate,▁▁▁▁▁
epoch/loss,█▃▂▁▁
epoch/val_accuracy,▁▆███
epoch/val_loss,█▃▂▁▁
epoch/accuracy,0.85989
epoch/epoch,4
epoch/learning_rate,0.001
epoch/loss,0.38856
epoch/val_accuracy,0.8748


---
## Part 3: Custom Training Loop with W&B

In [19]:
run = wandb.init(
    project='cmpe258-assignment4',
    name='custom-training-loop',
    config={
        'learning_rate': 0.001,
        'epochs': 5,
        'batch_size': 64,
        'hidden_units': [128, 64], # Added missing parameter
        'dropout_rate': 0.3      # Added missing parameter
    }
)

In [20]:
# Setup for custom training
model = create_model(wandb.config)
optimizer = keras.optimizers.Adam(learning_rate=wandb.config['learning_rate'])
loss_fn = keras.losses.SparseCategoricalCrossentropy()

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(10000).batch(64)
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(64)

# Metrics
train_loss = keras.metrics.Mean()
train_acc = keras.metrics.SparseCategoricalAccuracy()
val_loss = keras.metrics.Mean()
val_acc = keras.metrics.SparseCategoricalAccuracy()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [21]:
@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        predictions = model(x, training=True)
        loss = loss_fn(y, predictions)

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    train_loss.update_state(loss)
    train_acc.update_state(y, predictions)

    # Return gradient norm for logging
    grad_norm = tf.linalg.global_norm(gradients)
    return loss, grad_norm

In [22]:
# Custom training loop with detailed W&B logging
global_step = 0

for epoch in range(wandb.config['epochs']):
    train_loss.reset_state()
    train_acc.reset_state()

    for step, (x_batch, y_batch) in enumerate(train_ds):
        loss, grad_norm = train_step(x_batch, y_batch)
        global_step += 1

        # Log every 100 steps
        if step % 100 == 0:
            wandb.log({
                'batch_loss': loss.numpy(),
                'gradient_norm': grad_norm.numpy(),
                'learning_rate': optimizer.learning_rate.numpy(),
                'global_step': global_step
            })

    # Validation
    val_loss.reset_state()
    val_acc.reset_state()

    for x_batch, y_batch in val_ds:
        predictions = model(x_batch, training=False)
        val_loss.update_state(loss_fn(y_batch, predictions))
        val_acc.update_state(y_batch, predictions)

    # Log epoch metrics
    wandb.log({
        'epoch': epoch + 1,
        'train_loss': train_loss.result().numpy(),
        'train_accuracy': train_acc.result().numpy(),
        'val_loss': val_loss.result().numpy(),
        'val_accuracy': val_acc.result().numpy()
    })

    print(f"Epoch {epoch+1}: loss={train_loss.result():.4f}, acc={train_acc.result():.4f}, "
          f"val_loss={val_loss.result():.4f}, val_acc={val_acc.result():.4f}")

Epoch 1: loss=0.6894, acc=0.7584, val_loss=0.4141, val_acc=0.8458
Epoch 2: loss=0.4750, acc=0.8281, val_loss=0.3728, val_acc=0.8616
Epoch 3: loss=0.4345, acc=0.8428, val_loss=0.3491, val_acc=0.8704
Epoch 4: loss=0.4083, acc=0.8534, val_loss=0.3484, val_acc=0.8684
Epoch 5: loss=0.3912, acc=0.8605, val_loss=0.3371, val_acc=0.8764


In [23]:
wandb.finish()

batch_loss,█▃▃▂▃▂▂▂▂▂▁▂▂▂▂▂▃▂▂▂▁▁▂▂▁▂▁▂▂▂▂▂▁▂▂▁▁▂▂▂
epoch,▁▃▅▆█
global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
gradient_norm,█▄▅▄▅▄▅▃▃▄▂▄▄▄▃▃▃▃▄▃▂▃▃▃▂▅▁▃▅▆▃▄▃▃▃▁▂▃▃▄
learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_accuracy,▁▆▇██
train_loss,█▃▂▁▁
val_accuracy,▁▅▇▆█
val_loss,█▄▂▂▁
batch_loss,0.4428
epoch,5


---
## Part 4: Hyperparameter Sweeps

In [24]:
# Define sweep configuration
sweep_config = {
    'method': 'bayes',  # Options: grid, random, bayes
    'metric': {
        'name': 'val_accuracy',
        'goal': 'maximize'
    },
    'parameters': {
        'learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 1e-4,
            'max': 1e-2
        },
        'hidden_units_1': {
            'values': [64, 128, 256]
        },
        'hidden_units_2': {
            'values': [32, 64, 128]
        },
        'dropout_rate': {
            'distribution': 'uniform',
            'min': 0.1,
            'max': 0.5
        },
        'batch_size': {
            'values': [32, 64, 128]
        },
        'optimizer': {
            'values': ['adam', 'sgd', 'rmsprop']
        }
    }
}

In [25]:
def train_sweep():
    """Training function for sweep."""
    run = wandb.init()
    config = wandb.config

    # Build model based on sweep config
    model = keras.Sequential([
        layers.Flatten(input_shape=(28, 28)),
        layers.Dense(config.hidden_units_1, activation='relu'),
        layers.Dropout(config.dropout_rate),
        layers.Dense(config.hidden_units_2, activation='relu'),
        layers.Dropout(config.dropout_rate),
        layers.Dense(10, activation='softmax')
    ])

    # Select optimizer
    if config.optimizer == 'adam':
        opt = keras.optimizers.Adam(learning_rate=config.learning_rate)
    elif config.optimizer == 'sgd':
        opt = keras.optimizers.SGD(learning_rate=config.learning_rate, momentum=0.9)
    else:
        opt = keras.optimizers.RMSprop(learning_rate=config.learning_rate)

    model.compile(
        optimizer=opt,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(
        X_train, y_train,
        epochs=5,
        batch_size=config.batch_size,
        validation_data=(X_val, y_val),
        callbacks=[WandbMetricsLogger()],
        verbose=0
    )

    # Log final validation accuracy
    _, val_acc = model.evaluate(X_val, y_val, verbose=0)
    wandb.log({'val_accuracy': val_acc})

In [26]:
# Create sweep
sweep_id = wandb.sweep(sweep_config, project='cmpe258-assignment4')
print(f"Sweep ID: {sweep_id}")

Create sweep with ID: oxmypwca
Sweep URL: https://wandb.ai/shreram25-san-jose-state-university/cmpe258-assignment4/sweeps/oxmypwca
Sweep ID: oxmypwca


In [27]:
# Run sweep agent (limit to 10 runs for demo)
wandb.agent(sweep_id, function=train_sweep, count=10)

wandb: Agent Starting Run: 9mmd1501 with config:
wandb: 	batch_size: 32
wandb: 	dropout_rate: 0.4736182367171892
wandb: 	hidden_units_1: 256
wandb: 	hidden_units_2: 32
wandb: 	learning_rate: 0.0017624599360030293
wandb: 	optimizer: rmsprop
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


epoch/accuracy,▁▇▇██
epoch/epoch,▁▃▅▆█
epoch/learning_rate,▁▁▁▁▁
epoch/loss,█▁▁▁▂
epoch/val_accuracy,▁▆▄██
epoch/val_loss,▂▂▁▆█
val_accuracy,▁
epoch/accuracy,0.80476
epoch/epoch,4
epoch/learning_rate,0.00176
epoch/loss,0.66466


wandb: Agent Starting Run: ahd4lax2 with config:
wandb: 	batch_size: 64
wandb: 	dropout_rate: 0.20343138995652743
wandb: 	hidden_units_1: 64
wandb: 	hidden_units_2: 128
wandb: 	learning_rate: 0.0005559098541463005
wandb: 	optimizer: sgd
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


epoch/accuracy,▁▆▇▇█
epoch/epoch,▁▃▅▆█
epoch/learning_rate,▁▁▁▁▁
epoch/loss,█▃▂▁▁
epoch/val_accuracy,▁▄▆▇█
epoch/val_loss,█▅▃▂▁
val_accuracy,▁
epoch/accuracy,0.76753
epoch/epoch,4
epoch/learning_rate,0.00056
epoch/loss,0.66159


wandb: Agent Starting Run: cuu056kw with config:
wandb: 	batch_size: 32
wandb: 	dropout_rate: 0.4717779795224421
wandb: 	hidden_units_1: 64
wandb: 	hidden_units_2: 32
wandb: 	learning_rate: 0.0010281832706403762
wandb: 	optimizer: sgd
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


...

epoch/accuracy,▁▆▇██
epoch/epoch,▁▃▅▆█
epoch/learning_rate,▁▁▁▁▁
epoch/loss,█▃▂▁▁
epoch/val_accuracy,▄▁▇▂█
epoch/val_loss,▇█▃▇▁
val_accuracy,▁
epoch/accuracy,0.87175
epoch/epoch,4
epoch/learning_rate,0.00146
epoch/loss,0.35231


wandb: Agent Starting Run: nwk51vjb with config:
wandb: 	batch_size: 64
wandb: 	dropout_rate: 0.1070492916434272
wandb: 	hidden_units_1: 256
wandb: 	hidden_units_2: 32
wandb: 	learning_rate: 0.00021911357396346072
wandb: 	optimizer: rmsprop
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


epoch/accuracy,▁▆▇██
epoch/epoch,▁▃▅▆█
epoch/learning_rate,▁▁▁▁▁
epoch/loss,█▃▂▁▁
epoch/val_accuracy,▁▆▇██
epoch/val_loss,█▄▂▁▁
val_accuracy,▁
epoch/accuracy,0.86824
epoch/epoch,4
epoch/learning_rate,0.00022
epoch/loss,0.37062


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: 0up5qjgr with config:
wandb: 	batch_size: 64
wandb: 	dropout_rate: 0.12002700638540152
wandb: 	hidden_units_1: 256
wandb: 	hidden_units_2: 128
wandb: 	learning_rate: 0.0005336532649771687
wandb: 	optimizer: rmsprop
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


epoch/accuracy,▁▆▇▇█
epoch/epoch,▁▃▅▆█
epoch/learning_rate,▁▁▁▁▁
epoch/loss,█▃▂▂▁
epoch/val_accuracy,▁▁▆██
epoch/val_loss,██▂▁▁
val_accuracy,▁
epoch/accuracy,0.88449
epoch/epoch,4
epoch/learning_rate,0.00053
epoch/loss,0.31998


---
## Part 5: Model Artifacts and Versioning

In [28]:
run = wandb.init(
    project='cmpe258-assignment4',
    name='model-artifacts-demo',
    config=config
)

In [29]:
# Train model
model = create_model(wandb.config)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_val, y_val), verbose=1)

Epoch 1/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.7932 - loss: 0.5844 - val_accuracy: 0.8660 - val_loss: 0.3832
Epoch 2/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8490 - loss: 0.4133 - val_accuracy: 0.8700 - val_loss: 0.3493
Epoch 3/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8645 - loss: 0.3712 - val_accuracy: 0.8776 - val_loss: 0.3344
Epoch 4/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8704 - loss: 0.3520 - val_accuracy: 0.8670 - val_loss: 0.3670
Epoch 5/5
860/860 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8748 - loss: 0.3379 - val_accuracy: 0.8856 - val_loss: 0.3164


In [30]:
# Save model and log as artifact
model.save('fashion_mnist_model.keras')

# Create artifact
artifact = wandb.Artifact(
    name='fashion-mnist-mlp',
    type='model',
    description='MLP classifier for Fashion MNIST',
    metadata={
        'framework': 'keras',
        'architecture': 'MLP',
        'input_shape': '(28, 28)',
        'num_classes': 10
    }
)

# Add model file to artifact
artifact.add_file('fashion_mnist_model.keras')

# Log artifact
wandb.log_artifact(artifact)
print("Model artifact logged!")

Model artifact logged!


In [31]:
# Log dataset as artifact for reproducibility
dataset_artifact = wandb.Artifact(
    name='fashion-mnist-processed',
    type='dataset',
    description='Preprocessed Fashion MNIST dataset'
)

# Save dataset splits
np.savez('dataset.npz',
         X_train=X_train, y_train=y_train,
         X_val=X_val, y_val=y_val,
         X_test=X_test, y_test=y_test)

dataset_artifact.add_file('dataset.npz')
wandb.log_artifact(dataset_artifact)

<Artifact fashion-mnist-processed>

In [32]:
wandb.finish()

In [33]:
# Load model from artifact (in a new run)
run = wandb.init(project='cmpe258-assignment4', name='load-artifact-demo')

# Download artifact
artifact = run.use_artifact('fashion-mnist-mlp:latest')
artifact_dir = artifact.download()

# Load model
loaded_model = keras.models.load_model(f"{artifact_dir}/fashion_mnist_model.keras")

# Evaluate
test_loss, test_acc = loaded_model.evaluate(X_test, y_test, verbose=0)
print(f"Loaded model test accuracy: {test_acc:.4f}")

wandb.finish()

wandb:   1 of 1 files downloaded.  


Loaded model test accuracy: 0.8711


---
## Part 6: Alerts and Reports

In [35]:
run = wandb.init(project='cmpe258-assignment4', name='alerts-demo', config=config)

# Train model with alerts
model = create_model(wandb.config)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

for epoch in range(5):
    history = model.fit(X_train, y_train, epochs=1, batch_size=64,
                        validation_data=(X_val, y_val), verbose=0)

    val_acc = history.history['val_accuracy'][0]
    wandb.log({'val_accuracy': val_acc, 'epoch': epoch + 1})

    # Send alert if accuracy crosses threshold
    if val_acc > 0.88:
        wandb.alert(
            title='High Accuracy Achieved!',
            text=f'Validation accuracy reached {val_acc:.4f} at epoch {epoch+1}',
            level=wandb.AlertLevel.INFO
        )

    print(f"Epoch {epoch+1}: val_accuracy = {val_acc:.4f}")

wandb.finish()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1: val_accuracy = 0.8494
Epoch 2: val_accuracy = 0.8750
Epoch 3: val_accuracy = 0.8754
Epoch 4: val_accuracy = 0.8822
Epoch 5: val_accuracy = 0.8862


epoch,▁▃▅▆█
val_accuracy,▁▆▆▇█
epoch,5
val_accuracy,0.8862


---
## Summary

| Feature | Purpose | Key Functions |
|---------|---------|---------------|
| **wandb.init()** | Start experiment | project, name, config, tags |
| **wandb.log()** | Log metrics | Scalars, images, tables, plots |
| **WandbMetricsLogger** | Keras callback | Auto-log training metrics |
| **wandb.plot** | Visualizations | confusion_matrix, pr_curve, roc_curve |
| **wandb.sweep()** | HP optimization | Bayesian, random, grid search |
| **wandb.Artifact** | Model versioning | Save/load models, datasets |
| **wandb.alert()** | Notifications | Training alerts, thresholds |

**W&B Dashboard provides:**
- Real-time training visualization
- Experiment comparison
- Hyperparameter importance analysis
- Collaborative experiment tracking